# 餐厅销售数据分析 (Restaurant Sales Data EDA)

本notebook对餐厅销售数据进行清理和探索性数据分析

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示和图表风格
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# 加载数据
df = pd.read_csv('restaurant_sales_data.csv')
print(f"数据集大小: {df.shape[0]} 行, {df.shape[1]} 列")

数据集大小: 17534 行, 9 列


## 1. 数据概览

In [2]:
# 查看前几行数据
df.head(10)

,Order ID,Customer ID,Category,Item,Price,Quantity,Order Total,Order Date,Payment Method
0,ORD_705844,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,ORD_338528,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,ORD_443849,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,ORD_630508,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,ORD_648269,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash
5,ORD_381680,CUST_031,Main Dishes,Salmon,18.0,5.0,90.0,2022-07-20,Digital Wallet
6,ORD_270994,CUST_071,Side Dishes,Garlic Bread,4.0,5.0,20.0,2022-08-19,Credit Card
7,ORD_146656,CUST_077,Main Dishes,NaN,15.0,3.0,45.0,2023-02-15,Cash
8,ORD_428611,CUST_083,Desserts,NaN,6.0,2.0,12.0,2023-12-16,Cash
9,ORD_743636,CUST_085,Main Dishes,Vegetarian Platter,14.0,5.0,70.0,2022-08-07,NaN


In [ ]:
# 数据类型和基本信息
df.info()

In [ ]:
# 统计描述
df.describe()

## 2. 缺失值分析

In [ ]:
# 计算每列缺失值数量和百分比
missing_data = pd.DataFrame({
    '缺失数量': df.isnull().sum(),
    '缺失百分比': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['缺失数量'] > 0].sort_values('缺失数量', ascending=False)
print("缺失值统计:")
missing_data

In [ ]:
# 可视化缺失值
if len(missing_data) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    missing_data['缺失百分比'].plot(kind='bar', color='coral', ax=ax)
    ax.set_title('Missing Values by Column')
    ax.set_ylabel('Missing Percentage (%)')
    ax.set_xlabel('Column')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. 数据清理

In [ ]:
# 创建清理后的数据副本
df_clean = df.copy()

# 转换日期格式
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])

# 填充缺失的Item为'Unknown'
df_clean['Item'] = df_clean['Item'].fillna('Unknown')

# 填充缺失的Payment Method为'Unknown'
df_clean['Payment Method'] = df_clean['Payment Method'].fillna('Unknown')

# 填充缺失的Price（使用同Category的中位数）
df_clean['Price'] = df_clean.groupby('Category')['Price'].transform(
    lambda x: x.fillna(x.median())
)

# 填充缺失的Quantity（使用中位数）
df_clean['Quantity'] = df_clean['Quantity'].fillna(df_clean['Quantity'].median())

# 重新计算Order Total（如果有缺失）
df_clean['Order Total'] = df_clean['Price'] * df_clean['Quantity']

print("清理后缺失值:")
print(df_clean.isnull().sum())

In [ ]:
# 添加时间特征
df_clean['Year'] = df_clean['Order Date'].dt.year
df_clean['Month'] = df_clean['Order Date'].dt.month
df_clean['Day of Week'] = df_clean['Order Date'].dt.day_name()
df_clean['Quarter'] = df_clean['Order Date'].dt.quarter

df_clean.head()

## 4. 探索性数据分析

### 4.1 类别分析

In [ ]:
# 各类别订单数量
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 订单数量
category_counts = df_clean['Category'].value_counts()
axes[0].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Order Distribution by Category')

# 销售额
category_sales = df_clean.groupby('Category')['Order Total'].sum().sort_values(ascending=True)
category_sales.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Total Sales by Category')
axes[1].set_xlabel('Total Sales ($)')

plt.tight_layout()
plt.show()

In [ ]:
# 各类别统计
category_stats = df_clean.groupby('Category').agg({
    'Order ID': 'count',
    'Order Total': ['sum', 'mean'],
    'Quantity': 'sum',
    'Price': 'mean'
}).round(2)
category_stats.columns = ['订单数', '总销售额', '平均订单额', '总销量', '平均单价']
category_stats.sort_values('总销售额', ascending=False)

### 4.2 热销商品分析

In [ ]:
# Top 10 热销商品（按销量）
top_items_qty = df_clean[df_clean['Item'] != 'Unknown'].groupby('Item')['Quantity'].sum().nlargest(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_items_qty.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Top 10 Items by Quantity Sold')
axes[0].set_xlabel('Quantity')

# Top 10 商品（按销售额）
top_items_sales = df_clean[df_clean['Item'] != 'Unknown'].groupby('Item')['Order Total'].sum().nlargest(10)
top_items_sales.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Top 10 Items by Sales')
axes[1].set_xlabel('Sales ($)')

plt.tight_layout()
plt.show()

### 4.3 时间趋势分析

In [ ]:
# 月度销售趋势
monthly_sales = df_clean.groupby(df_clean['Order Date'].dt.to_period('M')).agg({
    'Order Total': 'sum',
    'Order ID': 'count'
})
monthly_sales.columns = ['Sales', 'Orders']

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.bar(range(len(monthly_sales)), monthly_sales['Sales'], color='steelblue', alpha=0.7, label='Sales')
ax1.set_xlabel('Month')
ax1.set_ylabel('Sales ($)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(range(len(monthly_sales)), monthly_sales['Orders'], color='red', marker='o', label='Orders')
ax2.set_ylabel('Number of Orders', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Monthly Sales and Orders Trend')
ax1.set_xticks(range(0, len(monthly_sales), 3))
ax1.set_xticklabels(monthly_sales.index.astype(str)[::3], rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 星期销售分布
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_sales = df_clean.groupby('Day of Week')['Order Total'].agg(['sum', 'mean', 'count'])
daily_sales = daily_sales.reindex(day_order)
daily_sales.columns = ['Total Sales', 'Avg Order Value', 'Order Count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

daily_sales['Total Sales'].plot(kind='bar', ax=axes[0], color='purple', alpha=0.7)
axes[0].set_title('Total Sales by Day of Week')
axes[0].set_ylabel('Sales ($)')
axes[0].tick_params(axis='x', rotation=45)

daily_sales['Order Count'].plot(kind='bar', ax=axes[1], color='green', alpha=0.7)
axes[1].set_title('Order Count by Day of Week')
axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4.4 支付方式分析

In [ ]:
# 支付方式分布
payment_stats = df_clean.groupby('Payment Method').agg({
    'Order ID': 'count',
    'Order Total': ['sum', 'mean']
})
payment_stats.columns = ['Order Count', 'Total Sales', 'Avg Order Value']
payment_stats = payment_stats.sort_values('Total Sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(payment_stats['Order Count'], labels=payment_stats.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Orders by Payment Method')

payment_stats['Avg Order Value'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Average Order Value by Payment Method')
axes[1].set_ylabel('Average Order Value ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

payment_stats.round(2)

### 4.5 客户分析

In [ ]:
# 客户消费分布
customer_stats = df_clean.groupby('Customer ID').agg({
    'Order ID': 'count',
    'Order Total': 'sum'
}).rename(columns={'Order ID': 'Order Count', 'Order Total': 'Total Spent'})

print(f"总客户数: {len(customer_stats)}")
print(f"\n客户消费统计:")
print(customer_stats.describe().round(2))

In [ ]:
# Top 10 客户
top_customers = customer_stats.nlargest(10, 'Total Spent')

fig, ax = plt.subplots(figsize=(10, 5))
top_customers['Total Spent'].plot(kind='bar', ax=ax, color='darkblue')
ax.set_title('Top 10 Customers by Total Spending')
ax.set_ylabel('Total Spent ($)')
ax.set_xlabel('Customer ID')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.6 价格和数量分布

In [ ]:
# 价格和数量分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_clean['Price'].hist(bins=20, ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')

df_clean['Quantity'].hist(bins=10, ax=axes[1], color='lightgreen', edgecolor='black')
axes[1].set_title('Quantity Distribution')
axes[1].set_xlabel('Quantity')
axes[1].set_ylabel('Frequency')

df_clean['Order Total'].hist(bins=30, ax=axes[2], color='salmon', edgecolor='black')
axes[2].set_title('Order Total Distribution')
axes[2].set_xlabel('Order Total ($)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 4.7 相关性分析

In [ ]:
# 数值变量相关性热力图
numeric_cols = ['Price', 'Quantity', 'Order Total']
correlation = df_clean[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. 数据汇总

In [ ]:
# 关键指标汇总
summary = {
    '总订单数': len(df_clean),
    '总销售额': f"${df_clean['Order Total'].sum():,.2f}",
    '平均订单额': f"${df_clean['Order Total'].mean():.2f}",
    '客户数': df_clean['Customer ID'].nunique(),
    '商品种类数': df_clean['Item'].nunique(),
    '类别数': df_clean['Category'].nunique(),
    '数据时间范围': f"{df_clean['Order Date'].min().date()} 至 {df_clean['Order Date'].max().date()}"
}

print("=" * 50)
print("数据汇总 (Data Summary)")
print("=" * 50)
for key, value in summary.items():
    print(f"{key}: {value}")

In [ ]:
# 保存清理后的数据（可选）
# df_clean.to_csv('restaurant_sales_data_cleaned.csv', index=False)
# print("清理后的数据已保存!")